# DES-PEP-CESD-001 — Phase 0: Protocol, estimand and analysis freeze

**Protocol §5.** This notebook does the Phase 0 *verification* work: it loads the
frozen freeze-artifacts, checks their internal consistency and cross-references,
confirms the frozen values against the protocol's own §5/§8/§14 constraints,
derives the resource forecast from the frozen ceiling, evaluates the **Gate 0
release matrix** mechanically, and writes a **SHA-256 manifest** of the evidence
package.

It does **not** sign the gate. PASS/HOLD/FAIL is a human reviewer decision
(§3.1); this notebook produces the evidence those reviewers act on.

Outputs consumed here (all authored as version-controlled freeze artifacts):
- `outputs/identifier_scheme.yaml`
- `outputs/question_estimand_registry.yaml`
- `outputs/scope_and_resource_ceiling.yaml`
- `../../sap/SAP_shell_v1.yaml`
- `../../governance/roles_and_arrangements.yaml`
- `../../governance/blinding_charter.yaml`
- `../../governance/deviation_log_template.yaml`
- `../../governance/gate_certificate_template.yaml`
- `../../governance/reviewer_checklists.yaml`

In [1]:
import os, sys, glob, json, hashlib, datetime
import yaml
import pandas as pd

# Resolve repo root robustly: walk UP from the current working directory until we
# find the directory that holds the campaign markers. This works regardless of
# where Jupyter was launched (repo root, the notebook folder, or in between) and
# under nbconvert/in-process execution.
def find_repo_root(start):
    markers = ("governance", "phases", "sap")
    p = os.path.abspath(start)
    while True:
        if all(os.path.isdir(os.path.join(p, m)) for m in markers):
            return p
        parent = os.path.dirname(p)
        if parent == p:            # reached filesystem root without a match
            raise FileNotFoundError(
                "Could not locate the des-pep-cesd-001 repo root (a dir containing "
                "governance/, phases/ and sap/) at or above: " + os.path.abspath(start)
                + ". Run this notebook from inside the cloned repo tree.")
        p = parent

REPO = find_repo_root(os.getcwd())
print("repo root:", REPO)
OUT = os.path.join(REPO, "phases", "phase00_freeze", "outputs")

def load_yaml(rel):
    path = os.path.join(REPO, rel)
    with open(path) as f:
        return yaml.safe_load(f), path

ids, ids_p     = load_yaml("phases/phase00_freeze/outputs/identifier_scheme.yaml")
qer, qer_p     = load_yaml("phases/phase00_freeze/outputs/question_estimand_registry.yaml")
scope, scope_p = load_yaml("phases/phase00_freeze/outputs/scope_and_resource_ceiling.yaml")
sap, sap_p     = load_yaml("sap/SAP_shell_v1.yaml")
roles, roles_p = load_yaml("governance/roles_and_arrangements.yaml")
blind, blind_p = load_yaml("governance/blinding_charter.yaml")

print("loaded 6 core freeze artifacts")

FileNotFoundError: Could not locate the des-pep-cesd-001 repo root (a dir containing governance/, phases/ and sap/) at or above: /Users/rossgibson/des-peptide-study/enhanced. Run this notebook from inside the cloned repo tree.

## 1. Consistency checks

Each check is a hard assertion with an explanatory message. A failure here means
the freeze artifacts disagree with each other or with the protocol's frozen
constants — which must be fixed before the gate can be reviewed, not signed around.

In [2]:
checks = []
def check(name, ok, detail=""):
    checks.append({"check": name, "result": "PASS" if ok else "FAIL", "detail": detail})
    return ok

# --- Protocol identity threads through every artifact ---
PID = "DES-PEP-CESD-001"
for nm, doc in [("identifier_scheme", ids), ("question_estimand", qer),
                ("scope_resource", scope), ("SAP_shell", sap),
                ("roles", roles), ("blinding", blind)]:
    check(f"protocol_id present in {nm}", doc.get("protocol_id") == PID,
          f"got {doc.get('protocol_id')!r}")

# --- Peptides consistent between id scheme and estimand roles ---
pep_codes = {p["code"] for p in ids["peptides"]}
check("three peptides registered (GGE, CME, YIY)",
      pep_codes == {"GGE", "CME", "YIY"}, f"{sorted(pep_codes)}")
check("GGE is the pilot/confirmatory-core peptide",
      any(p["code"] == "GGE" and "pilot" in p["role"].lower() for p in ids["peptides"]))

# --- Solvents consistent (three) and masked-arm namespace reserved ---
sol_codes = {s["code"] for s in ids["solvents"]}
check("three solvents registered (WATER, RELINE, GLYCELINE)",
      sol_codes == {"WATER", "RELINE", "GLYCELINE"}, f"{sorted(sol_codes)}")
check("masked-arm namespace has 3 arms",
      len(ids["masked_arms"]["namespace"]) == 3, f"{ids['masked_arms']['namespace']}")
check("solvent-code key NOT embedded in identifier scheme (hard firewall §18.3)",
      "mapping" not in {k.lower() for k in ids["masked_arms"]} or
      ids["masked_arms"].get("mapping_status", "").upper().startswith("TO BE ASSIGNED"),
      "mapping must be steward-held, not in analyst-readable freeze artifact")

# --- Start blocks: three strata, no solvent-effect basis (§8.1) ---
sb_codes = {b["code"] for b in ids["start_blocks"]}
check("three start-block strata (compact/intermediate/extended)",
      sb_codes == {"compact", "intermediate", "extended"}, f"{sorted(sb_codes)}")

# --- Sign convention frozen (§2.3) ---
sign = qer["primary_estimand"]["sign_convention"].lower()
check("ΔG sign convention frozen (positive = open less favourable)",
      "positive" in sign and "open" in sign and ("less favourable" in sign or "less favorable" in sign))

# --- Primary contrast is reline - water (§2.3) ---
prim = qer["contrasts"]["primary"]["expression"].lower().replace(" ", "")
check("primary contrast is ΔG_reline − ΔG_water",
      "reline" in prim and "water" in prim and "−" in qer["contrasts"]["primary"]["expression"])
check("two specificity contrasts prespecified",
      len(qer["contrasts"]["specificity"]) == 2)

# --- Claim hierarchy has exactly the five §16.4 outcome classes ---
classes = {c["class"] for c in qer["claim_hierarchy"]}
expected_classes = {"Confirmed solvent shift", "Small directional shift",
                    "Equivalent within margin", "Inconclusive", "Not supported / opposite"}
check("five outcome classes match §16.4", classes == expected_classes,
      f"missing={expected_classes - classes}, extra={classes - expected_classes}")

# --- Routes A/B/stop defined and decided at Gate 7 (§12.1) ---
route_ids = {r["id"] for r in scope["publication_routes"]}
check("Routes A, B and stop defined", route_ids == {"ROUTE-A", "ROUTE-B", "ROUTE-STOP"}, f"{route_ids}")
check("route decision deferred to Gate 7", scope["route_decision_gate"] == 7)
check("prohibited decision bases recorded (§12.1)",
      len(scope["prohibited_decision_basis"]) >= 3)

# --- Compute ceiling matches §5 task 4 frozen constants ---
cc = scope["compute_ceiling"]
check("max 400 ns per replica (§5 task 4)", cc["max_ns_per_enhanced_sampling_replica"] == 400,
      f"{cc['max_ns_per_enhanced_sampling_replica']}")
check("max 6 replicas per cell (§5 task 4)", cc["max_independent_replicas_per_cell"] == 6,
      f"{cc['max_independent_replicas_per_cell']}")
check("min 4 confirmatory replicas per cell (§8.1)", cc["min_confirmatory_replicas_per_cell"] == 4,
      f"{cc['min_confirmatory_replicas_per_cell']}")

# --- SAP shell: open reviewer-judgment items flagged SIGN:, not fabricated ---
margin = str(sap["equivalence"]["meaningful_margin_kJ_per_mol"])
check("equivalence margin left as SIGN: (not fabricated)", margin.strip().upper().startswith("SIGN:"),
      f"got {margin!r}")
check("primary model implementation left as SIGN: (frozen at Gate 7)",
      str(sap["across_replica_primary_model"]["implementation"]).strip().upper().startswith("SIGN:"))
check("SAP final freeze deferred to Gate 7", sap["final_freeze_gate"] == 7)

# --- Governance: six controlled roles with separated authority (§3.1) ---
role_names = {r["role"] for r in roles["controlled_roles"]}
expected_roles = {"Study lead", "Molecular-model reviewer", "Enhanced-sampling reviewer",
                  "Statistical reviewer", "Quantum-chemistry reviewer", "Data steward"}
check("six controlled roles present (§3.1)", role_names == expected_roles,
      f"missing={expected_roles - role_names}")

# --- Agent boundary: session will NOT sign gates / hold key / run production MD ---
wonts = " ".join(roles["agent_boundary"]["will_not"]).lower()
check("agent boundary: will not sign gates", "sign" in wonts and "gate" in wonts)
check("agent boundary: will not hold solvent key", "solvent-code key" in wonts or "solvent key" in wonts)
check("agent boundary: will not run production MD in sandbox", "production md" in wonts)

# --- Blinding: key steward-held, unblinding once after Gate 9/10 ---
check("solvent key owned by data steward", "steward" in blind["key_custody"]["owner_role"].lower())
check("unblinding occurs once after Gate 9 lock + Gate 10 freeze",
      "gate 9" in blind["unblinding_event"]["when"].lower() and "gate 10" in blind["unblinding_event"]["when"].lower())

checks_df = pd.DataFrame(checks)
n_fail = (checks_df["result"] == "FAIL").sum()
print(checks_df.to_string(index=False))
print(f"\nConsistency checks: {len(checks_df)-n_fail}/{len(checks_df)} PASS, {n_fail} FAIL")
assert n_fail == 0, "Consistency failures must be resolved before Gate 0 review"

                                                                   check result                                                                                                             detail
                                protocol_id present in identifier_scheme   PASS                                                                                             got 'DES-PEP-CESD-001'
                                protocol_id present in question_estimand   PASS                                                                                             got 'DES-PEP-CESD-001'
                                   protocol_id present in scope_resource   PASS                                                                                             got 'DES-PEP-CESD-001'
                                        protocol_id present in SAP_shell   PASS                                                                                             got 'DES-PEP-CESD-001'
                         

## 2. Resource forecast from the frozen ceiling

Derived from the frozen compute ceiling (§5 task 4) and the *planning* throughput
band (§19.1). These are **scheduling ranges, not guarantees** — Gate 4 replaces
them with the measured M5 Max benchmark (§9, §19). Reproduced here so the Gate 0
record shows the calendar implied by the ceiling the reviewers are freezing.

In [3]:
tp_lo, tp_hi = scope["planning_throughput"]["ns_per_day_range"]          # 60, 160
ov_lo, ov_hi = scope["planning_throughput"]["operational_overhead_fraction"]  # 0.15, 0.25

def forecast(total_ns):
    raw_lo = total_ns / tp_hi          # fastest throughput -> fewest days
    raw_hi = total_ns / tp_lo          # slowest throughput -> most days
    plan_lo = raw_lo * (1 + ov_lo)
    plan_hi = raw_hi * (1 + ov_hi)
    return raw_lo, raw_hi, plan_lo, plan_hi

scenarios = [
    ("GGE 50 ns pilot: 3 rep × 3 solv", 9 * 50),
    ("Route A: GGE 4 rep × 3 solv × 200 ns", 4 * 3 * 200),
    ("Route A precision: 6 rep", 6 * 3 * 200),
    ("Route B: 3 pep × 3 solv × 4 rep × 200 ns", 3 * 3 * 4 * 200),
    ("Route B precision: 6 rep", 3 * 3 * 6 * 200),
]
rows = []
for name, ns in scenarios:
    rlo, rhi, plo, phi = forecast(ns)
    rows.append({"scenario": name, "total_us": round(ns/1000, 2),
                 "raw_days": f"{rlo:.0f}–{rhi:.0f}",
                 "planning_days": f"{plo:.0f}–{phi:.0f}"})
forecast_df = pd.DataFrame(rows)
print(forecast_df.to_string(index=False))
forecast_df.to_csv(os.path.join(OUT, "resource_forecast.csv"), index=False)
print("\nwrote outputs/resource_forecast.csv")
print("NOTE: planning only; superseded by Gate 4 measured benchmark (§9.3, §19).")

                                scenario  total_us raw_days planning_days
         GGE 50 ns pilot: 3 rep × 3 solv      0.45      3–8           3–9
    Route A: GGE 4 rep × 3 solv × 200 ns      2.40    15–40         17–50
                Route A precision: 6 rep      3.60    22–60         26–75
Route B: 3 pep × 3 solv × 4 rep × 200 ns      7.20   45–120        52–150
                Route B precision: 6 rep     10.80   68–180        78–225

wrote outputs/resource_forecast.csv
NOTE: planning only; superseded by Gate 4 measured benchmark (§9.3, §19).


## 3. Gate 0 release matrix (mechanical evaluation)

The §5 Gate 0 matrix has four controlled criteria. This evaluates the *evidence
readiness* of each — i.e. that the required evidence artifact exists, is
non-empty and contains the expected content. Disposition remains **PASS/HOLD/FAIL
by a human reviewer**; a green row here means "evidence is present and internally
valid", not "approved".

In [4]:
def exists_nonempty(rel):
    p = os.path.join(REPO, rel)
    return os.path.isfile(p) and os.path.getsize(p) > 0

gate0 = [
    {"criterion": "Question and estimands unambiguous",
     "evidence": "phases/phase00_freeze/outputs/question_estimand_registry.yaml",
     "evidence_ready": exists_nonempty("phases/phase00_freeze/outputs/question_estimand_registry.yaml")
                       and bool(qer.get("primary_estimand")) and bool(qer.get("contrasts")),
     "reviewer_role": "Study lead + statistical reviewer"},
    {"criterion": "Blinding responsibilities assigned",
     "evidence": "governance/blinding_charter.yaml + roles_and_arrangements.yaml",
     "evidence_ready": exists_nonempty("governance/blinding_charter.yaml")
                       and exists_nonempty("governance/roles_and_arrangements.yaml")
                       and any(r["role"] == "Data steward" for r in roles["controlled_roles"]),
     "reviewer_role": "Data steward"},
    {"criterion": "Scope options and resource ceiling frozen",
     "evidence": "phases/phase00_freeze/outputs/scope_and_resource_ceiling.yaml",
     "evidence_ready": exists_nonempty("phases/phase00_freeze/outputs/scope_and_resource_ceiling.yaml")
                       and len(scope["publication_routes"]) == 3
                       and scope["compute_ceiling"]["max_ns_per_enhanced_sampling_replica"] == 400,
     "reviewer_role": "Study lead"},
    {"criterion": "Change control operational",
     "evidence": "governance/deviation_log_template.yaml + gate_certificate_template.yaml + repo tag",
     "evidence_ready": exists_nonempty("governance/deviation_log_template.yaml")
                       and exists_nonempty("governance/gate_certificate_template.yaml"),
     "reviewer_role": "Data steward"},
]
gate0_df = pd.DataFrame(gate0)
gate0_df["disposition"] = gate0_df["evidence_ready"].map(lambda b: "READY FOR REVIEW" if b else "EVIDENCE INCOMPLETE")
print(gate0_df[["criterion", "evidence_ready", "disposition", "reviewer_role"]].to_string(index=False))
all_ready = bool(gate0_df["evidence_ready"].all())
print(f"\nAll Gate 0 evidence ready for reviewer sign-off: {all_ready}")

                                criterion  evidence_ready      disposition                     reviewer_role
       Question and estimands unambiguous            True READY FOR REVIEW Study lead + statistical reviewer
       Blinding responsibilities assigned            True READY FOR REVIEW                      Data steward
Scope options and resource ceiling frozen            True READY FOR REVIEW                        Study lead
               Change control operational            True READY FOR REVIEW                      Data steward

All Gate 0 evidence ready for reviewer sign-off: True


## 4. SHA-256 evidence manifest

Every Phase 0 artifact is hashed (§18.2: "Use SHA-256 for immutable scientific
artifacts and record the hash in the run registry before downstream analysis").
The manifest is written to `manifests/gate00_manifest.sha256` and is the object
the Gate 0 certificate references.

In [5]:
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

manifest_targets = [
    "phases/phase00_freeze/outputs/identifier_scheme.yaml",
    "phases/phase00_freeze/outputs/question_estimand_registry.yaml",
    "phases/phase00_freeze/outputs/scope_and_resource_ceiling.yaml",
    "phases/phase00_freeze/outputs/resource_forecast.csv",
    "sap/SAP_shell_v1.yaml",
    "governance/roles_and_arrangements.yaml",
    "governance/blinding_charter.yaml",
    "governance/deviation_log_template.yaml",
    "governance/gate_certificate_template.yaml",
    "governance/reviewer_checklists.yaml",
    "runs/run_registry.csv",
]
man_rows = []
for rel in manifest_targets:
    p = os.path.join(REPO, rel)
    man_rows.append({"sha256": sha256(p), "bytes": os.path.getsize(p), "path": rel})
man_df = pd.DataFrame(man_rows)

os.makedirs(os.path.join(REPO, "manifests"), exist_ok=True)
man_path = os.path.join(REPO, "manifests", "gate00_manifest.sha256")
with open(man_path, "w") as f:
    for r in man_rows:
        f.write(f"{r['sha256']}  {r['path']}\n")
print(man_df.to_string(index=False))
print(f"\nwrote {os.path.relpath(man_path, REPO)}")

                                                          sha256  bytes                                                          path
cc2357f82f4113a1ceb963a669707b05e561b6ca26ae14688137b3b66412f316   4424          phases/phase00_freeze/outputs/identifier_scheme.yaml
ce6c3ba4aac483bb89d8bd7e5cf5049771a4fa3b04b30b68edf20106f1015df5   4752 phases/phase00_freeze/outputs/question_estimand_registry.yaml
f3b233f0f1098ea033ee6f8af15b90eb39e58d94cc0b3965d51c205b56eed54b   4255 phases/phase00_freeze/outputs/scope_and_resource_ceiling.yaml
daef55c03a9f36b11ed5df88378a5f7fc425d79bb40dccbb46c64d15ecde2252    309           phases/phase00_freeze/outputs/resource_forecast.csv
11aec36ea8f14a4ff6d863ab2ead5ae3ddd1acc7ad7d4c0e685575347e08ee1c   4636                                         sap/SAP_shell_v1.yaml
f4844f09cbf2f7506622a60549f5825103578dd6e31ac1b7a0298950fb2a8d5c   4177                        governance/roles_and_arrangements.yaml
258f147e32c4d17478aa1a9162d0fa3f70f79ae5e416e41583a5e6d0389973

## 5. Gate 0 evidence summary (machine-readable)

A single JSON the reviewers and the (human-signed) gate certificate reference.

In [6]:
summary = {
    "protocol_id": PID,
    "gate": 0,
    "phase": "Phase 0 — Protocol, estimand and analysis freeze",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "consistency_checks": {"total": int(len(checks_df)),
                           "passed": int((checks_df["result"] == "PASS").sum()),
                           "failed": int(n_fail)},
    "gate0_release_matrix": gate0_df[["criterion", "evidence_ready", "disposition", "reviewer_role"]].to_dict("records"),
    "all_evidence_ready": all_ready,
    "manifest": {"file": "manifests/gate00_manifest.sha256", "artifact_count": len(man_rows)},
    "arrangements": {
        "production_md": scope["execution_platform"]["production_md"],
        "governance": "Human reviewers sign gates; data steward holds solvent key; session prepares evidence and enforces sequencing.",
    },
    "signing_status": "AWAITING HUMAN REVIEWER SIGN-OFF (not signed by the session)",
}
sum_path = os.path.join(OUT, "gate00_evidence_summary.json")
with open(sum_path, "w") as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))
print(f"\nwrote {os.path.relpath(sum_path, REPO)}")

{
  "protocol_id": "DES-PEP-CESD-001",
  "gate": 0,
  "phase": "Phase 0 \u2014 Protocol, estimand and analysis freeze",
  "generated_utc": "2026-07-14T16:06:25.395464+00:00",
  "consistency_checks": {
    "total": 31,
    "passed": 31,
    "failed": 0
  },
  "gate0_release_matrix": [
    {
      "criterion": "Question and estimands unambiguous",
      "evidence_ready": true,
      "disposition": "READY FOR REVIEW",
      "reviewer_role": "Study lead + statistical reviewer"
    },
    {
      "criterion": "Blinding responsibilities assigned",
      "evidence_ready": true,
      "disposition": "READY FOR REVIEW",
      "reviewer_role": "Data steward"
    },
    {
      "criterion": "Scope options and resource ceiling frozen",
      "evidence_ready": true,
      "disposition": "READY FOR REVIEW",
      "reviewer_role": "Study lead"
    },
    {
      "criterion": "Change control operational",
      "evidence_ready": true,
      "disposition": "READY FOR REVIEW",
      "reviewer_role": "Da

---
**Phase 0 verification complete.** All freeze artifacts are internally consistent,
frozen values match the protocol's §5/§8/§14 constants, open reviewer-judgment
items are flagged `SIGN:` rather than fabricated, and the evidence package is
hashed. The Gate 0 certificate (`governance/gate_certificate_template.yaml`)
remains to be **signed by the human reviewers**; once signed, the repo is tagged
`gate-00` and committed (§18.2).